# UC-PROC-3 — Asset-Scoped Geometry Validation

**Persona:** Data Engineer

**Goal:** After uploading a GeoParquet asset, list the asset-scoped process tasks,
trigger the geometry validation task, poll for completion, and retrieve the validation
report.

**Prerequisites:**
- Asset already uploaded via the assets notebook (`assets/01`) — `ASSET_ID` must be known
- Geometry validation process registered and available as an asset-scoped task
- Write-capable JWT in `DYNASTORE_WRITE_TOKEN`

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_WRITE_TOKEN`, `CATALOG_ID`,
`COLLECTION_ID`, `ASSET_ID`

In [ ]:
import os
import json
import time

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["DYNASTORE_BASE_URL"]
WRITE_TOKEN = os.environ["DYNASTORE_WRITE_TOKEN"]
CATALOG_ID = os.environ["CATALOG_ID"]
COLLECTION_ID = os.environ["COLLECTION_ID"]
ASSET_ID = os.environ["ASSET_ID"]

headers = {
    "Authorization": f"Bearer {WRITE_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

job_id = None
TASK_ID = None

print(f"Connected to {BASE_URL}")
print(f"CATALOG_ID={CATALOG_ID}  COLLECTION_ID={COLLECTION_ID}  ASSET_ID={ASSET_ID}")

In [ ]:
# List asset-scoped tasks available for this asset
r = client.get(f"/assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}/tasks")
print(r.status_code)
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

body = r.json()
tasks = body if isinstance(body, list) else body.get("tasks", body.get("items", []))
assert len(tasks) > 0, (
    f"No asset-scoped tasks found for asset {ASSET_ID}. "
    f"Ensure the geometry validation process is registered."
)

print(f"Available tasks ({len(tasks)}):")
for task in tasks:
    task_id = task.get("id", task.get("name", "?"))
    title = task.get("title", task.get("summary", ""))
    print(f"  {task_id}: {title}")

# Select the geometry validation task
validation_tasks = [
    t for t in tasks
    if "geom" in t.get("id", "").lower() or "valid" in t.get("id", "").lower()
]
assert validation_tasks, (
    f"No geometry validation task found in asset tasks: {[t.get('id') for t in tasks]}"
)
TASK_ID = validation_tasks[0]["id"]
print(f"\nSelected TASK_ID={TASK_ID}")

In [ ]:
# Submit the geometry validation task
task_payload = {
    "inputs": {
        "validate_topology": True,
        "fix_invalid": False,
    }
}

r = client.post(
    f"/assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}/tasks/{TASK_ID}/execute",
    content=json.dumps(task_payload),
)
print(r.status_code, r.text[:400])
assert r.status_code == 201, (
    f"Expected 201 (job created), got {r.status_code}: {r.text}"
)

receipt = r.json()
job_id = receipt.get("jobID", receipt.get("job_id", ""))
assert job_id, f"No jobID in task execution response: {receipt}"
print(f"Validation task submitted: job_id={job_id}")

In [ ]:
# Poll until the job reaches a terminal status
status = "accepted"
for attempt in range(20):
    resp = client.get(f"/processes/jobs/{job_id}")
    assert resp.status_code == 200, (
        f"Expected 200 polling job, got {resp.status_code}: {resp.text}"
    )
    status = resp.json().get("status", "")
    print(f"Attempt {attempt + 1}: status={status}")
    if status in ("successful", "failed", "dismissed"):
        break
    time.sleep(min(2 ** attempt, 30))

assert status == "successful", (
    f"Validation job did not reach 'successful'; final status='{status}'"
)
print(f"Geometry validation completed: status={status}")

In [ ]:
# Retrieve and print the validation report
r = client.get(f"/processes/jobs/{job_id}/results")
print(r.status_code)
assert r.status_code == 200, f"Expected 200 fetching results, got {r.status_code}: {r.text}"

report = r.json()
print("Validation report:")
print(json.dumps(report, indent=2)[:1200])

In [ ]:
# Dismiss the completed job
r = client.delete(f"/processes/jobs/{job_id}")
print(r.status_code)
assert r.status_code == 204, (
    f"Expected 204 dismissing job, got {r.status_code}: {r.text}"
)
print(f"Job {job_id} dismissed.")

## Edge Cases

In [ ]:
# Edge case 1: asset not in collection -> 404
# If ASSET_ID does not belong to CATALOG_ID, the asset-tasks endpoint returns 404.
# The platform does not leak cross-tenant asset existence via error shape.
fake_asset_id = "00000000-0000-0000-0000-000000000000"
r = client.get(f"/assets/catalogs/{CATALOG_ID}/assets/{fake_asset_id}/tasks")
print(r.status_code)
assert r.status_code == 404, (
    f"Expected 404 for unknown asset, got {r.status_code}: {r.text}"
)
print("404 confirmed: non-existent asset correctly returns 404.")

In [ ]:
# Edge case 2: task incompatible with asset type -> 422 (schema validation failure)
# If a task's input schema requires a specific asset type (e.g. GeoParquet) but the
# asset is a plain CSV, the execution endpoint validates the asset media type against
# the process descriptor and returns 422 Unprocessable Entity before starting the job.
#
# Example: submitting a raster-only task against a vector asset:
#   POST /assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}/tasks/raster-statistics/execute
#   -> 422 { "detail": "Asset media type 'application/geoparquet' is incompatible with "
#             "task input schema: expected 'image/tiff; application=geotiff'" }
#
# Use a known-incompatible task id to trigger 422:
incompatible_task_id = "raster-band-statistics"  # vector asset -> raster task
r = client.post(
    f"/assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}/tasks/{incompatible_task_id}/execute",
    content=json.dumps({"inputs": {}}),
)
print(r.status_code)
assert r.status_code in (404, 422), (
    f"Expected 404 (task not found) or 422 (incompatible type), "
    f"got {r.status_code}: {r.text}"
)
print(
    f"{r.status_code} confirmed for incompatible task: "
    f"{'task not registered for this asset type' if r.status_code == 404 else 'schema validation rejected the request'}."
)

In [ ]:
# Edge case 3: collection-scoped variant
# Asset-scoped tasks can also be invoked at the collection level when the target is
# an entire collection rather than a single asset. The URL form is longer and the
# inputs may accept an optional asset filter.
#
# Collection-scoped validation URL pattern:
#
#   POST /assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}
#             /assets/{ASSET_ID}/tasks/{TASK_ID}/execute
#
# This is semantically equivalent to the shorter form but the platform uses the
# COLLECTION_ID segment to resolve the correct schema and apply collection-level
# write policies before dispatching the task.
#
# Verify the longer form returns the same task listing:
if TASK_ID:
    r_coll = client.get(
        f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
        f"/assets/{ASSET_ID}/tasks"
    )
    print(r_coll.status_code)
    assert r_coll.status_code in (200, 404), (
        f"Expected 200 or 404 for collection-scoped task list, "
        f"got {r_coll.status_code}: {r_coll.text}"
    )
    if r_coll.status_code == 200:
        coll_tasks = r_coll.json()
        coll_task_list = (
            coll_tasks if isinstance(coll_tasks, list)
            else coll_tasks.get("tasks", coll_tasks.get("items", []))
        )
        coll_task_ids = [t.get("id", "") for t in coll_task_list]
        print(f"Collection-scoped task ids: {coll_task_ids}")
        assert TASK_ID in coll_task_ids, (
            f"TASK_ID '{TASK_ID}' not present in collection-scoped listing: {coll_task_ids}"
        )
        print("Collection-scoped URL form confirmed equivalent.")
    else:
        print(
            "404: collection-scoped task route not mounted in this deployment.\n"
            "Use the shorter /assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}/tasks form."
        )
else:
    print("TASK_ID not resolved; skipping collection-scoped variant check.")

client.close()